# 12 - Global gap-fill pipeline (native 4 km, few days, 1 variable)

A first working whole-globe run of the gap-fill U-Net at native 4 km resolution, on a few recent days of
Copernicus GlobColour CHL. It is meant to bring the pipeline to life at full resolution rather than to be a
tuned product, so it holds only a few days and one variable.

**Design decisions baked in:**

1. **Land-heavy tiles** are dropped by a low ocean-fraction threshold. Land is masked out of the loss anyway,
   so this is an efficiency filter, and the threshold is kept low so coastal tiles survive.
2. **Poles** use the native lat/lon grid with no reprojection. The geo x, y, z channels give the model its
   position on the sphere, and the domain is clipped to plus or minus 80 degrees to drop the worst distortion
   and the mostly-empty polar rows. Reprojection to an equal-area grid is a later refinement.
3. **Tile size follows the artifact calculation.** With synthetic clouds at blob_sigma 20 px (about 90 km),
   the safe tile is at least about ten times that, so the tiles target about 240 px, roughly 12x the cloud,
   above the ten-times rule and well under the roughly 1000 px GPU cap. The grid does not divide evenly at
   exactly 240, so the divisor logic lands on a nearby rectangle (256 by 240 on the 10-day grid), which is
   Eli's point in action. A small batch (8) keeps the resize-conv model within the T4's memory.
4. **Ocean-but-no-data tiles** (polar night, persistent cloud) are skipped by a minimum observed-data
   fraction, which also removes the empty high-latitude tiles.
5. **The date line wraps.** Longitude tiles wrap around the antimeridian so the model sees east-west
   continuity, and because the geo channels are x, y, z on the sphere they are continuous across that seam.

**Rectangular tiles (Eli's point):** square tiles that do not divide the grid force overlap. Here the tile
height and width are each chosen to divide the grid dimensions, latitude and longitude independently, so the
tiles partition the globe with no forced overlap in training. Prediction still uses overlapping tiles with a
Hann blend, since blending needs overlap to remove seams.

**Streaming:** the full 4 km global channel cube is far too large to hold, so only the selected tiles are
materialized. The raw chlorophyll cube (a few days, about 0.4 GB) stays in memory and each tile's channels are
built on the fly.

## Setup

This Hub drops pip installs on kernel restart, so this cell reinstalls the two packages if they are missing
and imports everything the later cells use. Run it once after any restart.

In [ ]:
import sys, subprocess, importlib
def ensure(mod, spec, extra=()):
    try:
        importlib.import_module(mod)
    except ImportError:
        print("installing", spec, "...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, spec], check=True)
        importlib.invalidate_caches(); importlib.import_module(mod)
ensure("mindthegap", "git+https://github.com/SAFS-Varanasi-Internship/mindthegap.git@troy-branch",
       ["--force-reinstall", "--no-cache-dir"])
ensure("copernicusmarine", "copernicusmarine")

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf, mindthegap as mtg
for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)
bpc = mtg.build_pace_channels
WORK = os.path.expanduser("~/shared-public/mindthegap")
CACHE = f"{WORK}/cache"; MODELS = f"{WORK}/models"
os.makedirs(CACHE, exist_ok=True); os.makedirs(MODELS, exist_ok=True)
print("ready")

## Read native 4 km global, a few recent days

Downloads full-resolution global CHL for a few recent days (the dense multi-sensor era, well past 2010),
clips to plus or minus 80 degrees, log-transforms, and caches. This is the slow, one-time step, roughly
0.15 GB downloaded per day, so it is cached to disk and reused on later runs. Land comes from the product flag.
`NDAYS = 10` pulls enough data to train; lower it for a quick smoke test.

In [ ]:
NDAYS = 10; LATCLIP = 80.0
START, END = "2024-10-01", "2024-12-31"       # recent dense multi-sensor era (post-2010, as late as available)
GCACHE = f"{CACHE}/global_native_n{NDAYS}_lat{int(LATCLIP)}.npz"

if os.path.exists(GCACHE):
    z = np.load(GCACHE, allow_pickle=True)
    chl_log = z["chl_log"]; times = z["times"]; gLAT = z["gLAT"]; gLON = z["gLON"]; gocean = z["gocean"]
    print("reused", GCACHE, chl_log.shape)
else:
    import copernicusmarine
    ds = copernicusmarine.open_dataset(
        dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D",
        variables=["CHL", "flags"],
        minimum_latitude=-LATCLIP, maximum_latitude=LATCLIP, start_datetime=START, end_datetime=END)
    ds = ds.rename({c: n for c, n in [("latitude", "lat"), ("longitude", "lon")] if c in ds.coords})
    chl = ds["CHL"].isel(time=slice(-NDAYS, None))
    gh, gw = chl.sizes["lat"], chl.sizes["lon"]
    cy = slice(0, gh - gh % 8); cx = slice(0, gw - gw % 8); chl = chl.isel(lat=cy, lon=cx)
    times = chl.time.values; gH, gW = chl.sizes["lat"], chl.sizes["lon"]
    gLAT = chl.lat.values; gLON = chl.lon.values; T = chl.sizes["time"]
    chl_log = np.empty((T, gH, gW), np.float32)
    for t in range(T):
        day = chl.isel(time=t).values                 # one native 4 km global day (~150 MB)
        chl_log[t] = np.log(np.where(day > 0, day, np.nan)).astype("float32")
        print(f"  read {t+1}/{T}", flush=True)
    land = (ds["flags"].isel(time=0) == 1).isel(lat=cy, lon=cx).values[:gH, :gW]
    gocean = (~land) & np.isfinite(chl_log).any(0)
    try:
        np.savez(GCACHE, chl_log=chl_log, times=times, gLAT=gLAT, gLON=gLON, gocean=gocean)
        print("saved", GCACHE, f"({os.path.getsize(GCACHE)/1e9:.2f} GB)")
    except OSError as e:
        print("cache save skipped:", e)

gT, gH, gW = chl_log.shape
gext = [float(gLON.min()), float(gLON.max()), float(gLAT.min()), float(gLAT.max())]
covm = float(np.mean([np.isfinite(chl_log[t])[gocean].mean() for t in range(gT)]))
print(f"native global {gH}x{gW} (4 km), {gT} frames | ocean {gocean.mean():.0%} | mean coverage {covm:.0%}")

## Tile plan, filters, and training

The tile height and width are each the divisor of the grid dimension closest to the target size, so the tiles
partition the grid exactly (no forced overlap), and longitude wraps at the date line. Tiles that are mostly
land or have almost no observations are dropped. The kept tiles are built once and cached to disk, so a crash
does not force a rebuild. Training follows the streamlined pipeline: the target is the full field with plain
mean squared error, the upsampler is resize-then-conv (bilinear) rather than a transposed convolution so there
is no checkerboard, and a checkpoint saves the best model to disk as it trains.

In [ ]:
BLOB = 20; COVERAGE_SC = 0.25            # cloud sigma in 4 km px (~90 km); the tile below is ~12x this
OCEAN_MIN = 0.15; DATACOV_MIN = 0.05     # decisions 1 and 4
MAX_TILES = 1200; BATCH = 8              # batch 8 keeps the resize-conv model within T4 memory

def fit_tile(dim, target=256, floor=10*BLOB):
    # divisor of `dim` (multiple of 8) closest to target, preferring the >= floor band -> exact partition
    divs = [d for d in range(8, dim + 1, 8) if dim % d == 0]
    band = [d for d in divs if floor <= d <= target * 1.5]
    cands = band if band else divs
    return min(cands, key=lambda d: abs(d - target))
TH, TW = fit_tile(gH), fit_tile(gW)
print(f"tiles {TH}x{TW}  (lat {gH}/{TH}={gH//TH}, lon {gW}/{TW}={gW//TW})  min ratio {min(TH, TW)/BLOB:.1f}x blob")

grng = np.random.default_rng(0); gp = grng.permutation(gT)
gtr = np.zeros(gT, bool); gtr[gp[:max(1, int(round(0.67 * gT)))]] = True
trf = list(np.where(gtr)[0]); vaf = list(np.where(~gtr)[0]) or trf

CS = 8; sub = np.ascontiguousarray(chl_log[:, ::CS, ::CS])
_, _, STATS, ORDER = bpc(sub, times, gtr, n_days=1, cloud_mode="synthetic", coverage=COVERAGE_SC,
                         blob_sigma=max(2, BLOB // CS), time_sigma=1.0, seed=0, land=~gocean[::CS, ::CS])
gm0, gs0 = STATS["CHL"]; NCg = len(ORDER) + 3

def wrapcols(b): return (b + np.arange(TW)) % gW
def geo_of(a, cols):
    la = np.deg2rad(gLAT[a:a+TH])[:, None]; lo = np.deg2rad(gLON[cols])[None, :]
    return np.stack([np.broadcast_to(np.cos(la)*np.cos(lo), (TH, TW)),
                     np.broadcast_to(np.cos(la)*np.sin(lo), (TH, TW)),
                     np.broadcast_to(np.sin(la), (TH, TW))], -1).astype("float32")
def build_tile(a, b, d, seed, cov=COVERAGE_SC):
    c = wrapcols(b); cc = np.ascontiguousarray(chl_log[:, a:a+TH][:, :, c])
    ch, y, _, _ = bpc(cc, times, gtr, n_days=1, cloud_mode="synthetic", coverage=cov,
                      blob_sigma=BLOB, time_sigma=1.0, seed=seed, stats=STATS, land=~gocean[a:a+TH][:, c])
    X = np.concatenate([np.stack([ch[k] for k in ORDER], -1)[d], geo_of(a, c)], -1).astype("float32")
    Y = np.nan_to_num(y[d], nan=0.0).astype("float32")[..., None]   # full target, plain MSE (Eli's setup)
    return X, Y

lat_starts = list(range(0, gH, TH))          # exact partition in lat (TH divides gH)
if lat_starts[-1] != gH - TH: lat_starts.append(gH - TH)
lon_starts = list(range(0, gW, TW))          # lon wraps; last tile meets the first with no overlap when TW | gW
gpos = []
for a in lat_starts:
    for b in lon_starts:
        c = wrapcols(b); toc = gocean[a:a+TH][:, c]
        if toc.mean() < OCEAN_MIN: continue
        if np.isfinite(chl_log[:, a:a+TH][:, :, c])[:, toc].mean() < DATACOV_MIN: continue
        gpos.append((a, b))
print(f"{len(gpos)} tiles kept of {len(lat_starts)*len(lon_starts)}")

# Build the training tiles once and cache them to disk, so a crash does not force a rebuild.
TILECACHE = f"{CACHE}/tiles_{TH}x{TW}_n{gT}.npz"
if os.path.exists(TILECACHE):
    z = np.load(TILECACHE); Xtr, Ytr, Xv, Yv = z["Xtr"], z["Ytr"], z["Xv"], z["Yv"]
    print("loaded cached tiles", Xtr.shape, "from disk")
else:
    rb = np.random.default_rng(3)
    pairs = [(a, b, d) for (a, b) in gpos for d in trf]
    if len(pairs) > MAX_TILES: pairs = [pairs[i] for i in rb.permutation(len(pairs))[:MAX_TILES]]
    Xtr = np.empty((len(pairs), TH, TW, NCg), np.float32); Ytr = np.empty((len(pairs), TH, TW, 1), np.float32)
    for i, (a, b, d) in enumerate(pairs):
        Xtr[i], Ytr[i] = build_tile(a, b, d, 1000 + i)
        if (i + 1) % 80 == 0: print(f"  built {i+1}/{len(pairs)}", flush=True)
    vs = gpos[::max(1, len(gpos) // 12)][:12]
    Xv = np.stack([build_tile(a, b, vaf[0], 7)[0] for (a, b) in vs])
    Yv = np.stack([build_tile(a, b, vaf[0], 7)[1] for (a, b) in vs])
    try:
        np.savez(TILECACHE, Xtr=Xtr, Ytr=Ytr, Xv=Xv, Yv=Yv)
        print("cached tiles", f"({os.path.getsize(TILECACHE)/1e9:.1f} GB)")
    except OSError as e:
        print("tile cache save skipped (disk):", e)
print("train tiles:", Xtr.shape, f"~{Xtr.nbytes/1e9:.1f} GB")

def UNet_rc(input_shape):                     # resize-conv U-Net: bilinear upsample + conv, no transpose checkerboard
    from tensorflow.keras import Input, layers
    inp = Input(shape=input_shape); x = inp; filters = [64, 128, 256]; enc = []
    for f in filters:
        enc.append(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x); x = layers.BatchNormalization()(x)
    for f, e in zip(filters[:-1][::-1], enc[::-1][:-1]):
        x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.concatenate([x, e])
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
    x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
    x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    x = layers.concatenate([x, enc[0]])
    x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
    out = layers.Conv2D(1, 3, padding="same", activation="linear")(x)
    return tf.keras.Model(inp, out, name="U-net-rc")

MPATH = f"{MODELS}/global_native_rc_t{TH}x{TW}.keras"
tf.keras.backend.clear_session()
gmodel = UNet_rc((None, None, NCg))
gmodel.compile(tf.keras.optimizers.Adam(3e-4, clipnorm=1.0), loss="mse", metrics=["mae"], jit_compile=False)
# ModelCheckpoint saves the best model to disk as it improves, so an interruption keeps progress.
cbs = [tf.keras.callbacks.ModelCheckpoint(MPATH, monitor="val_loss", save_best_only=True, verbose=0),
       tf.keras.callbacks.EarlyStopping("val_loss", patience=15, restore_best_weights=True)]
gmodel.fit(Xtr, Ytr, batch_size=BATCH, validation_data=(Xv, Yv), epochs=150, callbacks=cbs, verbose=2, shuffle=True)
gmodel.save(MPATH)
p = gmodel.predict(Xv[:8], verbose=0)[..., 0]
print(f"saved | val pred range [{p.min():+.2f}, {p.max():+.2f}]  (sane is about [-4, 1])")

## Recovery after a crash (optional)

If the kernel dies, run the setup cell (for the imports) and then this cell, and skip straight to predict. It
reloads the cached cube, rebuilds the deterministic setup, and loads the checkpointed model, with no
retraining. To keep training instead, run the training cell, which loads the cached tiles and continues.

In [ ]:
NDAYS = 10; LATCLIP = 80.0; BLOB = 20; COVERAGE_SC = 0.25
z = np.load(f"{CACHE}/global_native_n{NDAYS}_lat{int(LATCLIP)}.npz", allow_pickle=True)
chl_log = z["chl_log"]; times = z["times"]; gLAT = z["gLAT"]; gLON = z["gLON"]; gocean = z["gocean"]
gT, gH, gW = chl_log.shape
gext = [float(gLON.min()), float(gLON.max()), float(gLAT.min()), float(gLAT.max())]
def fit_tile(dim, target=256, floor=10*BLOB):
    divs = [d for d in range(8, dim + 1, 8) if dim % d == 0]
    band = [d for d in divs if floor <= d <= target * 1.5]
    return min(band if band else divs, key=lambda d: abs(d - target))
TH, TW = fit_tile(gH), fit_tile(gW)
grng = np.random.default_rng(0); gp = grng.permutation(gT)
gtr = np.zeros(gT, bool); gtr[gp[:max(1, int(round(0.67 * gT)))]] = True
trf = list(np.where(gtr)[0]); vaf = list(np.where(~gtr)[0]) or trf
CS = 8; sub = np.ascontiguousarray(chl_log[:, ::CS, ::CS])
_, _, STATS, ORDER = bpc(sub, times, gtr, n_days=1, cloud_mode="synthetic", coverage=COVERAGE_SC,
                         blob_sigma=max(2, BLOB // CS), time_sigma=1.0, seed=0, land=~gocean[::CS, ::CS])
gm0, gs0 = STATS["CHL"]; NCg = len(ORDER) + 3
gmodel = tf.keras.models.load_model(f"{MODELS}/global_native_rc_t{TH}x{TW}.keras", compile=False)
print(f"recovered | grid {gH}x{gW} | tiles {TH}x{TW} | model loaded, ready to predict")

## Predict the whole globe and composite

Longitude is padded by half a tile with wrapped columns so the date line is continuous, the padded frame is
covered with overlapping tiles blended by a Hann window, and the result is cropped back. The composite keeps
the real observations and shows the model only in the gaps, which is how a gap-fill product is used.

In [ ]:
def predict_frame(model, d):
    PADW = TW // 2
    chl_p = np.concatenate([chl_log[:, :, -PADW:], chl_log, chl_log[:, :, :PADW]], axis=2)
    oce_p = np.concatenate([gocean[:, -PADW:], gocean, gocean[:, :PADW]], axis=1)
    lon_p = np.concatenate([gLON[-PADW:], gLON, gLON[:PADW]])     # geo uses cos/sin, periodic, so raw wrap is fine
    Hh, Wp = gH, gW + 2 * PADW
    ovh, ovw = TH // 4, TW // 4; sh, sw = TH - ovh, TW - ovw
    lat_s = list(range(0, Hh - TH + 1, sh)) or [0]
    if lat_s[-1] != Hh - TH: lat_s.append(Hh - TH)
    lon_s = list(range(0, Wp - TW + 1, sw)) or [0]
    if lon_s[-1] != Wp - TW: lon_s.append(Wp - TW)
    wgt = (np.outer(np.hanning(TH), np.hanning(TW)) + 1e-3).astype("float32")
    acc = np.zeros((Hh, Wp), np.float32); cnt = np.zeros((Hh, Wp), np.float32)
    for a in lat_s:
        Xs = []
        for b in lon_s:
            cc = np.ascontiguousarray(chl_p[:, a:a+TH, b:b+TW])
            ch, _, _, _ = bpc(cc, times, gtr, n_days=1, cloud_mode="synthetic", coverage=0.0,
                              blob_sigma=BLOB, time_sigma=1.0, seed=0, stats=STATS, land=~oce_p[a:a+TH, b:b+TW])
            # GAPFILL relabel (Eli's fix): real gaps become the estimate target, like the synthetic clouds the
            # model was trained to fill; clear the "unavailable" flag. Without this the model estimates nothing.
            ch["fake_cloud_flag"] = ch["real_cloud_flag"].astype("float32")
            ch["real_cloud_flag"] = np.zeros_like(ch["real_cloud_flag"])
            la = np.deg2rad(gLAT[a:a+TH])[:, None]; lo = np.deg2rad(lon_p[b:b+TW])[None, :]
            g = np.stack([np.broadcast_to(np.cos(la)*np.cos(lo), (TH, TW)),
                          np.broadcast_to(np.cos(la)*np.sin(lo), (TH, TW)),
                          np.broadcast_to(np.sin(la), (TH, TW))], -1).astype("float32")
            Xs.append(np.concatenate([np.stack([ch[k] for k in ORDER], -1)[d], g], -1).astype("float32"))
        preds = model.predict(np.stack(Xs), batch_size=16, verbose=0)[..., 0] * gs0 + gm0
        for b, p in zip(lon_s, preds):
            acc[a:a+TH, b:b+TW] += p * wgt; cnt[a:a+TH, b:b+TW] += wgt
    return (acc / np.maximum(cnt, 1e-6))[:, PADW:PADW+gW]

d = int(vaf[0])
pred = predict_frame(gmodel, d)
gorigin = "upper" if gLAT[0] > gLAT[-1] else "lower"     # Copernicus lat is ascending -> "lower" (north up)
import matplotlib.patches as mpatches
observed = gocean & np.isfinite(chl_log[d])
comp = np.where(observed, chl_log[d], np.where(gocean, pred, np.nan))
vmn, vmx = np.nanpercentile(chl_log[d][observed], [2, 98])
land_rgba = np.zeros((gH, gW, 4)); land_rgba[~gocean] = [0.55, 0.55, 0.55, 1.0]

ZH, ZW = min(240, gH), min(312, gW); step = 200          # auto-pick a structured, well-observed ocean box
best, i0, j0 = -1.0, 0, 0
for a in range(0, gH - ZH + 1, step):
    for b in range(0, gW - ZW + 1, step):
        m = observed[a:a+ZH, b:b+ZW]
        if m.mean() < 0.4: continue
        v = float(np.nanvar(np.where(m, chl_log[d, a:a+ZH, b:b+ZW], np.nan)))
        if v > best: best, i0, j0 = v, a, b
i1, j1 = i0 + ZH, j0 + ZW
zlat, zlon = gLAT[i0:i1], gLON[j0:j1]
zext = [float(zlon.min()), float(zlon.max()), float(zlat.min()), float(zlat.max())]

glob = [("observed (real gaps white)", np.where(observed, chl_log[d], np.nan)),
        ("composite: observed + model in the gaps", comp),
        ("model everywhere (raw)", np.where(gocean, pred, np.nan))]
fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(3, 2, width_ratios=[2.6, 1.0], hspace=0.14, wspace=0.05, left=0.03, right=0.9)
im = None
for r, (ttl, arr) in enumerate(glob):
    axg = fig.add_subplot(gs[r, 0])
    im = axg.imshow(arr, cmap="viridis", vmin=vmn, vmax=vmx, extent=gext, origin=gorigin, interpolation="nearest")
    axg.imshow(land_rgba, extent=gext, origin=gorigin, interpolation="nearest")
    axg.add_patch(mpatches.Rectangle((zext[0], zext[2]), zext[1]-zext[0], zext[3]-zext[2],
                                     fill=False, edgecolor="white", lw=1.4))
    axg.set_title(ttl, size=10); axg.set_xticks([]); axg.set_yticks([])
    axz = fig.add_subplot(gs[r, 1])
    axz.imshow(arr[i0:i1, j0:j1], cmap="viridis", vmin=vmn, vmax=vmx, extent=zext, origin=gorigin, interpolation="nearest")
    axz.imshow(land_rgba[i0:i1, j0:j1], extent=zext, origin=gorigin, interpolation="nearest")
    axz.set_title("zoom (native pixels)" if r == 0 else "", size=10); axz.set_xticks([]); axz.set_yticks([])
sm = plt.cm.ScalarMappable(norm=plt.Normalize(vmn, vmx), cmap="viridis"); sm.set_array([])
cax = fig.add_axes([0.92, 0.3, 0.012, 0.4]); fig.colorbar(sm, cax=cax, label="log Chl-a")
plt.suptitle(f"Global CHL gap-fill, native 4 km, {str(pd.to_datetime(times[d]).date())}   (white box = zoom)", y=0.99)
try:
    plt.savefig("global_native_composite.png", dpi=130, bbox_inches="tight")
except OSError as e:
    print("savefig skipped:", e)
plt.show()

## Notes

- The read is the heavy step, roughly 0.15 GB per day at native 4 km, so `NDAYS` is small by default. Raise it
  for a better-trained model, at the cost of more download.
- With only a few days the temporal prev/next channels are thin, so this leans on the spatial fill. It is the
  pipeline coming to life at full resolution, not a tuned product.
- Building each tile runs the channel builder with a wide gaussian, so the tile-building step and the
  whole-globe predict each take a few minutes. Both are one-time.
- Next milestones: more days (or 8-day composites) for coverage and a real temporal signal, a held-out score
  against persistence and climatology, and eventually an equal-area treatment of the poles.